In [2]:
#Goal: Create a model that can predict the profit of the company based on company's spending
# pattern and company's location

In [3]:
import pandas as pd
import numpy as np

In [4]:
np.set_printoptions(suppress=True)

In [5]:
data = pd.read_csv("50_Startups.csv")

In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   R&D Spend        50 non-null     float64
 1   Administration   50 non-null     float64
 2   Marketing Spend  50 non-null     float64
 3   State            50 non-null     object 
 4   Profit           50 non-null     float64
dtypes: float64(4), object(1)
memory usage: 2.1+ KB


In [7]:
data.State.unique()

array(['New York', 'California', 'Florida'], dtype=object)

In [8]:
#Seperate features and label

features = data.iloc[:,[0,1,2,3]].values
label = data.iloc[:,[4]].values

In [9]:
#OHE on state column using sklearn

from sklearn.preprocessing import OneHotEncoder
oheState = OneHotEncoder(sparse_output=False)
stateDummy = oheState.fit_transform(features[:,3].reshape(-1,1))

In [10]:
finalFeatureSet = np.concatenate((stateDummy,features[:,[0,1,2]]), axis = 1)
finalFeatureSet

array([[0.0, 0.0, 1.0, 165349.2, 136897.8, 471784.1],
       [1.0, 0.0, 0.0, 162597.7, 151377.59, 443898.53],
       [0.0, 1.0, 0.0, 153441.51, 101145.55, 407934.54],
       [0.0, 0.0, 1.0, 144372.41, 118671.85, 383199.62],
       [0.0, 1.0, 0.0, 142107.34, 91391.77, 366168.42],
       [0.0, 0.0, 1.0, 131876.9, 99814.71, 362861.36],
       [1.0, 0.0, 0.0, 134615.46, 147198.87, 127716.82],
       [0.0, 1.0, 0.0, 130298.13, 145530.06, 323876.68],
       [0.0, 0.0, 1.0, 120542.52, 148718.95, 311613.29],
       [1.0, 0.0, 0.0, 123334.88, 108679.17, 304981.62],
       [0.0, 1.0, 0.0, 101913.08, 110594.11, 229160.95],
       [1.0, 0.0, 0.0, 100671.96, 91790.61, 249744.55],
       [0.0, 1.0, 0.0, 93863.75, 127320.38, 249839.44],
       [1.0, 0.0, 0.0, 91992.39, 135495.07, 252664.93],
       [0.0, 1.0, 0.0, 119943.24, 156547.42, 256512.92],
       [0.0, 0.0, 1.0, 114523.61, 122616.84, 261776.23],
       [1.0, 0.0, 0.0, 78013.11, 121597.55, 264346.06],
       [0.0, 0.0, 1.0, 94657.16, 145077.58

# 1. Correlation Analysis

In [11]:
#Extract correlations

finalDataSetDF = pd.concat([pd.get_dummies(data.State, dtype=int), data.iloc[:,[0,1,2,4]]], axis = 1)
finalDataSetDF.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   California       50 non-null     int64  
 1   Florida          50 non-null     int64  
 2   New York         50 non-null     int64  
 3   R&D Spend        50 non-null     float64
 4   Administration   50 non-null     float64
 5   Marketing Spend  50 non-null     float64
 6   Profit           50 non-null     float64
dtypes: float64(4), int64(3)
memory usage: 2.9 KB


In [12]:
finalDataSetDF.corr()

,California,Florida,New York,R&D Spend,Administration,Marketing Spend,Profit
California,1.000000,-0.492366,-0.515152,-0.143165,-0.015478,-0.168875,-0.145837
Florida,-0.492366,1.000000,-0.492366,0.105711,0.010493,0.205685,0.116244
New York,-0.515152,-0.492366,1.000000,0.039068,0.005145,-0.033670,0.031368
R&D Spend,-0.143165,0.105711,0.039068,1.000000,0.241955,0.724248,0.972900
Administration,-0.015478,0.010493,0.005145,0.241955,1.000000,-0.032154,0.200717
Marketing Spend,-0.168875,0.205685,-0.033670,0.724248,-0.032154,1.000000,0.747766
Profit,-0.145837,0.116244,0.031368,0.972900,0.200717,0.747766,1.000000


In [13]:
#Guideline by Prashant N
#
# Select those feature columns whose corr is greater than equal to +/-0.5

In [14]:
# Selected Features are: rdSpend and markSpend
# Label: Profit
#
# Build Model and check model satisfies by Genralization Criteria




#2. Backward Elimination using OLS (Ordinary Least Square)

In [18]:
#1. Perform All IN
allInFeatures = np.append( np.ones( (len(finalFeatureSet), 1) ).astype(int), finalFeatureSet, axis = 1)

In [19]:
allInFeatures

array([[1, 0.0, 0.0, 1.0, 165349.2, 136897.8, 471784.1],
       [1, 1.0, 0.0, 0.0, 162597.7, 151377.59, 443898.53],
       [1, 0.0, 1.0, 0.0, 153441.51, 101145.55, 407934.54],
       [1, 0.0, 0.0, 1.0, 144372.41, 118671.85, 383199.62],
       [1, 0.0, 1.0, 0.0, 142107.34, 91391.77, 366168.42],
       [1, 0.0, 0.0, 1.0, 131876.9, 99814.71, 362861.36],
       [1, 1.0, 0.0, 0.0, 134615.46, 147198.87, 127716.82],
       [1, 0.0, 1.0, 0.0, 130298.13, 145530.06, 323876.68],
       [1, 0.0, 0.0, 1.0, 120542.52, 148718.95, 311613.29],
       [1, 1.0, 0.0, 0.0, 123334.88, 108679.17, 304981.62],
       [1, 0.0, 1.0, 0.0, 101913.08, 110594.11, 229160.95],
       [1, 1.0, 0.0, 0.0, 100671.96, 91790.61, 249744.55],
       [1, 0.0, 1.0, 0.0, 93863.75, 127320.38, 249839.44],
       [1, 1.0, 0.0, 0.0, 91992.39, 135495.07, 252664.93],
       [1, 0.0, 1.0, 0.0, 119943.24, 156547.42, 256512.92],
       [1, 0.0, 0.0, 1.0, 114523.61, 122616.84, 261776.23],
       [1, 1.0, 0.0, 0.0, 78013.11, 121597.55, 264

In [20]:
#Step2: Decide the SL of the project

SL = 0.05

In [22]:
#Step3: Perform OLS

# endog ---- label column
# exog ----- feature column

import statsmodels.regression.linear_model as stat

olsFormula = stat.OLS(endog=label , exog=allInFeatures.astype(float)).fit()
olsFormula.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.951
Model:                            OLS   Adj. R-squared:                  0.945
Method:                 Least Squares   F-statistic:                     169.9
Date:                Sat, 25 Apr 2026   Prob (F-statistic):           1.34e-27
Time:                        04:51:50   Log-Likelihood:                -525.38
No. Observations:                  50   AIC:                             1063.
Df Residuals:                      44   BIC:                             1074.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       3.763e+04   5073.636      7.417      0.000    2.74e+04    4.79e+04
x1          1.249e+04   2449.797      5.099      0.000    7554.868    1.74e+04
x2          1.269e+04   2726.700      4.654      0.000    7195.596    1.82e+04
x3          1.245e+04   2486.364      5.007      0.000    7439.285    1.75e+04
x4             0.8060      0.046     17.369      0.000       0.712       0.900
x5            -0.0270      0.052     -0.517      0.608      -0.132       0.078
x6             0.0270      0.017      1.574      0.123      -0.008       0.062
==============================================================================
Omnibus:                       14.782   Durbin-Watson:                   1.283
Prob(Omnibus):                  0.001   Jarque-Bera (JB):               21.266
Skew:                          -0.948   Prob(JB):                     2.41e-05
Kurtosis:                       5.572   Cond. No.                     1.81e+17
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 1.19e-22. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

In [23]:
#Step4: Select the feature column that has the highest p-value

# Selected feature is x5(adminSpend) ------ 0.608

In [24]:
#Step5: Check the condition
#
# if pvalue > SL:
#      eliminate feature and recreate new feature set
# else:
#      Go to step7

In [25]:
# Since pvalue is greater than 0.05, therefore eliminate AdminSpend(x5)

newFeatureSet = allInFeatures[:,[0,1,2,3,4,6]]

In [26]:
olsFormula = stat.OLS(endog=label , exog=newFeatureSet.astype(float)).fit()
olsFormula.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.950
Model:                            OLS   Adj. R-squared:                  0.946
Method:                 Least Squares   F-statistic:                     215.8
Date:                Sat, 25 Apr 2026   Prob (F-statistic):           9.72e-29
Time:                        04:56:08   Log-Likelihood:                -525.53
No. Observations:                  50   AIC:                             1061.
Df Residuals:                      45   BIC:                             1071.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       3.525e+04   2100.376     16.782      0.000     3.1e+04    3.95e+04
x1          1.171e+04   1910.312      6.130      0.000    7861.854    1.56e+04
x2          1.185e+04   2170.903      5.459      0.000    7477.785    1.62e+04
x3          1.169e+04   1988.428      5.879      0.000    7684.996    1.57e+04
x4             0.7967      0.042     18.771      0.000       0.711       0.882
x5             0.0298      0.016      1.842      0.072      -0.003       0.062
==============================================================================
Omnibus:                       14.640   Durbin-Watson:                   1.257
Prob(Omnibus):                  0.001   Jarque-Bera (JB):               21.037
Skew:                          -0.938   Prob(JB):                     2.70e-05
Kurtosis:                       5.565   Cond. No.                     1.46e+17
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 1.53e-22. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

In [27]:
# Select x5(Marketing Spend)
# Since x5 greater than 0.05, therefore eliminate
newFeatureSet = allInFeatures[:,[0,1,2,3,4]]
olsFormula = stat.OLS(endog=label , exog=newFeatureSet.astype(float)).fit()
olsFormula.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.947
Model:                            OLS   Adj. R-squared:                  0.943
Method:                 Least Squares   F-statistic:                     272.4
Date:                Sat, 25 Apr 2026   Prob (F-statistic):           2.76e-29
Time:                        04:57:27   Log-Likelihood:                -527.35
No. Observations:                  50   AIC:                             1063.
Df Residuals:                      46   BIC:                             1070.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       3.686e+04   1959.786     18.806      0.000    3.29e+04    4.08e+04
x1          1.189e+04   1956.677      6.079      0.000    7955.697    1.58e+04
x2          1.306e+04   2122.665      6.152      0.000    8785.448    1.73e+04
x3           1.19e+04   2036.022      5.847      0.000    7805.580     1.6e+04
x4             0.8530      0.030     28.226      0.000       0.792       0.914
==============================================================================
Omnibus:                       13.418   Durbin-Watson:                   1.122
Prob(Omnibus):                  0.001   Jarque-Bera (JB):               17.605
Skew:                          -0.907   Prob(JB):                     0.000150
Kurtosis:                       5.271   Cond. No.                     3.20e+17
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The smallest eigenvalue is 3.66e-24. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

In [ ]:
#FinalFeatureSet ---> California, Florida, NY, rdSpend
#label -------------> Profit

# Build model , check quality and if found approve and deploy the model